# AlphaBot Live Data Streams

This notebook turns the external data inputs from `alphabot.py` into live-updating plots.

- Weather updates from Open-Meteo
- Thames tide updates from the UK flood-monitoring API
- Optional Heathrow snapshot updates when `AERODATABOX_KEY` is set

Run the setup cells once, then run the final cell to start the live dashboard. Interrupt the kernel to stop it.


In [ ]:
import math
import os
import time
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

import matplotlib.pyplot as plt
import pandas as pd
import requests
from IPython.display import clear_output, display

try:
    import dotenv
except ModuleNotFoundError:
    dotenv = None

if dotenv is not None:
    dotenv.load_dotenv()

plt.style.use("seaborn-v0_8-whitegrid")

LONDON_TZ = ZoneInfo("Europe/London")
LONDON_LAT = 51.5074
LONDON_LON = -0.1278
THAMES_MEASURE = "0006-level-tidal_level-i-15_min-mAOD"
TIDE_CYCLE_SECS = 12 * 3600 + 25 * 60
AERODATABOX_KEY = os.getenv("AERODATABOX_KEY")


In [ ]:
def next_settlement_time(now=None):
    now = now or datetime.now(LONDON_TZ)
    settle = now.replace(hour=12, minute=0, second=0, microsecond=0)
    if now >= settle:
        settle += timedelta(days=1)
    return settle


def normalize_london_times(values):
    times = pd.to_datetime(values)
    tz = times.dt.tz if hasattr(times, "dt") else times.tz
    if tz is None:
        return times.dt.tz_localize("Europe/London") if hasattr(times, "dt") else times.tz_localize("Europe/London")
    return times.dt.tz_convert("Europe/London") if hasattr(times, "dt") else times.tz_convert("Europe/London")


def fit_tide_cycle(times, levels):
    sample_count = min(len(times), len(levels), 128)
    if sample_count < 16:
        return None

    fit_times = times[-sample_count:]
    fit_levels = levels[-sample_count:]
    anchor = fit_times[-1]
    omega = 2.0 * math.pi / TIDE_CYCLE_SECS

    n = float(sample_count)
    sum_sin = 0.0
    sum_cos = 0.0
    sum_sin2 = 0.0
    sum_cos2 = 0.0
    sum_sin_cos = 0.0
    sum_y = 0.0
    sum_y_sin = 0.0
    sum_y_cos = 0.0

    for stamp, level in zip(fit_times, fit_levels):
        phase = omega * (stamp - anchor).total_seconds()
        sin_v = math.sin(phase)
        cos_v = math.cos(phase)
        sum_sin += sin_v
        sum_cos += cos_v
        sum_sin2 += sin_v * sin_v
        sum_cos2 += cos_v * cos_v
        sum_sin_cos += sin_v * cos_v
        sum_y += level
        sum_y_sin += level * sin_v
        sum_y_cos += level * cos_v

    det = (
        n * (sum_sin2 * sum_cos2 - sum_sin_cos * sum_sin_cos)
        - sum_sin * (sum_sin * sum_cos2 - sum_sin_cos * sum_cos)
        + sum_cos * (sum_sin * sum_sin_cos - sum_sin2 * sum_cos)
    )
    if abs(det) < 1e-9:
        return None

    det_offset = (
        sum_y * (sum_sin2 * sum_cos2 - sum_sin_cos * sum_sin_cos)
        - sum_sin * (sum_y_sin * sum_cos2 - sum_sin_cos * sum_y_cos)
        + sum_cos * (sum_y_sin * sum_sin_cos - sum_sin2 * sum_y_cos)
    )
    det_sin = (
        n * (sum_y_sin * sum_cos2 - sum_sin_cos * sum_y_cos)
        - sum_y * (sum_sin * sum_cos2 - sum_sin_cos * sum_cos)
        + sum_cos * (sum_sin * sum_y_cos - sum_y_sin * sum_cos)
    )
    det_cos = (
        n * (sum_sin2 * sum_y_cos - sum_y_sin * sum_sin_cos)
        - sum_sin * (sum_sin * sum_y_cos - sum_y_sin * sum_cos)
        + sum_y * (sum_sin * sum_sin_cos - sum_sin2 * sum_cos)
    )

    return {
        "offset": det_offset / det,
        "sin_coeff": det_sin / det,
        "cos_coeff": det_cos / det,
        "omega": omega,
        "anchor_ts": anchor.timestamp(),
    }


def predict_tide_level(model, target):
    anchor = datetime.fromtimestamp(model["anchor_ts"], tz=target.tzinfo)
    phase = model["omega"] * (target - anchor).total_seconds()
    return model["offset"] + model["sin_coeff"] * math.sin(phase) + model["cos_coeff"] * math.cos(phase)


def project_tide_swing(model, start, end):
    if end <= start:
        return 0.0

    points = [start]
    stamp = start
    while True:
        stamp += timedelta(minutes=15)
        if stamp >= end:
            break
        points.append(stamp)
    if points[-1] != end:
        points.append(end)

    swing_sum = 0.0
    previous = predict_tide_level(model, points[0])
    for stamp in points[1:]:
        current = predict_tide_level(model, stamp)
        diff_cm = abs(current - previous) * 100.0
        swing_sum += max(0.0, 20.0 - diff_cm) + max(0.0, diff_cm - 25.0)
        previous = current
    return swing_sum


def future_projection_points(start, end, step_minutes=15):
    if end <= start:
        return [start]

    points = [start]
    stamp = start
    while True:
        stamp += timedelta(minutes=step_minutes)
        if stamp >= end:
            break
        points.append(stamp)
    if points[-1] != end:
        points.append(end)
    return points


In [ ]:
def fetch_weather_snapshot():
    settle = next_settlement_time()
    window_start = settle - timedelta(hours=24)

    resp = requests.get(
        "https://api.open-meteo.com/v1/forecast",
        params={
            "latitude": LONDON_LAT,
            "longitude": LONDON_LON,
            "minutely_15": "temperature_2m,relative_humidity_2m",
            "past_minutely_15": 96,
            "forecast_minutely_15": 96,
            "timezone": "Europe/London",
        },
        timeout=10,
    )
    resp.raise_for_status()
    raw = resp.json()["minutely_15"]

    weather_df = pd.DataFrame(
        {
            "time": normalize_london_times(raw["time"]),
            "temp_c": raw["temperature_2m"],
            "humidity": raw["relative_humidity_2m"],
        }
    )
    weather_df["temp_f"] = weather_df["temp_c"] * 9.0 / 5.0 + 32.0
    weather_df["wx_stream"] = weather_df["temp_f"] * weather_df["humidity"]
    weather_df["in_window"] = weather_df["time"].between(window_start, settle)
    weather_df["wx_sum_contrib"] = weather_df["wx_stream"].where(weather_df["in_window"], 0.0) / 100.0
    weather_df["wx_sum_running"] = weather_df["wx_sum_contrib"].cumsum()

    settle_idx = (weather_df["time"] - settle).abs().idxmin()
    metrics = {
        "wx_spot_now": float(weather_df.iloc[len(weather_df) // 2]["wx_stream"]),
        "wx_spot_settle": float(weather_df.loc[settle_idx, "wx_stream"]),
        "wx_sum": float(weather_df["wx_sum_contrib"].sum()),
        "settle": settle,
        "window_start": window_start,
    }
    return weather_df, metrics


def fetch_tide_snapshot():
    settle = next_settlement_time()
    window_start = settle - timedelta(hours=24)

    resp = requests.get(
        f"https://environment.data.gov.uk/flood-monitoring/id/measures/{THAMES_MEASURE}/readings",
        params={"_sorted": "", "_limit": 193},
        timeout=30,
    )
    resp.raise_for_status()
    items = [item for item in resp.json().get("items", []) if item.get("value") is not None]
    items.sort(key=lambda item: item["dateTime"])

    tide_df = pd.DataFrame(
        {
            "time": [
                datetime.fromisoformat(item["dateTime"].replace("Z", "+00:00")).astimezone(LONDON_TZ)
                for item in items
            ],
            "level_m": [float(item["value"]) for item in items],
        }
    )
    tide_df["abs_level_mm"] = tide_df["level_m"].abs() * 1000.0
    tide_df["diff_cm"] = tide_df["level_m"].diff().abs() * 100.0
    tide_df["swing_payoff"] = tide_df["diff_cm"].apply(
        lambda diff_cm: 0.0 if pd.isna(diff_cm) else max(0.0, 20.0 - diff_cm) + max(0.0, diff_cm - 25.0)
    )
    tide_df["in_window"] = tide_df["time"].between(window_start, settle)
    tide_df["swing_payoff_window"] = tide_df["swing_payoff"].where(tide_df["in_window"], 0.0)
    tide_df["swing_running"] = tide_df["swing_payoff_window"].cumsum()

    settle_proxy = settle - timedelta(hours=24)
    settle_idx = (tide_df["time"] - settle_proxy).abs().idxmin()
    latest_level = float(tide_df.iloc[-1]["level_m"])
    settle_level = 0.7 * float(tide_df.loc[settle_idx, "level_m"]) + 0.3 * latest_level
    tide_model = fit_tide_cycle(tide_df["time"].tolist(), tide_df["level_m"].tolist())

    projection_df = None
    projected_settle_level = None
    projected_swing_sum = None
    if tide_model is not None:
        projected_settle_level = 0.75 * predict_tide_level(tide_model, settle) + 0.25 * settle_level
        projected_swing_sum = project_tide_swing(tide_model, window_start, settle)
        projection_points = future_projection_points(tide_df["time"].iloc[-1], settle)
        projection_df = pd.DataFrame({"time": projection_points})
        projection_df["level_m"] = projection_df["time"].apply(
            lambda stamp: predict_tide_level(tide_model, stamp.to_pydatetime() if hasattr(stamp, "to_pydatetime") else stamp)
        )

    metrics = {
        "latest_level": latest_level,
        "tide_spot_raw": abs(settle_level) * 1000.0,
        "tide_spot_projected": None if projected_settle_level is None else abs(projected_settle_level) * 1000.0,
        "observed_swing_sum": float(tide_df["swing_payoff_window"].sum()),
        "projected_swing_sum": projected_swing_sum,
        "settle": settle,
        "window_start": window_start,
    }
    return tide_df, projection_df, metrics


def fetch_flight_snapshot():
    if not AERODATABOX_KEY:
        return None, None

    now = datetime.now(LONDON_TZ).replace(second=0, microsecond=0)
    start = now - timedelta(hours=12)
    resp = requests.get(
        f"https://aerodatabox.p.rapidapi.com/flights/airports/iata/LHR/{start:%Y-%m-%dT%H:%M}/{now:%Y-%m-%dT%H:%M}",
        params={"direction": "Both"},
        headers={
            "x-rapidapi-host": "aerodatabox.p.rapidapi.com",
            "x-rapidapi-key": AERODATABOX_KEY,
        },
        timeout=15,
    )
    resp.raise_for_status()
    payload = resp.json()
    arrivals = payload.get("arrivals", [])
    departures = payload.get("departures", [])

    rows = []
    for side, flights in (("arrival", arrivals), ("departure", departures)):
        for flight in flights:
            schedule = (flight.get("movement", {}) or {}).get("scheduledTime", {}) or {}
            local = schedule.get("local")
            if not local:
                continue
            rows.append({"side": side, "time": pd.Timestamp(local)})

    flights_df = pd.DataFrame(rows)
    if flights_df.empty:
        return flights_df, {"count": float(len(arrivals) + len(departures)), "start": start, "end": now}

    flights_df["time"] = normalize_london_times(flights_df["time"])
    flights_df["hour"] = flights_df["time"].dt.floor("1h")
    metrics = {"count": float(len(arrivals) + len(departures)), "start": start, "end": now}
    return flights_df, metrics


In [ ]:
def render_dashboard():
    weather_df, weather_metrics = fetch_weather_snapshot()
    tide_df, projection_df, tide_metrics = fetch_tide_snapshot()
    flights_df, flight_metrics = fetch_flight_snapshot()

    has_flights = flight_metrics is not None
    rows = 3 if has_flights else 2
    fig, axes = plt.subplots(rows, 1, figsize=(15, 5 * rows), constrained_layout=True)
    if rows == 1:
        axes = [axes]

    weather_ax = axes[0]
    weather_ax.plot(weather_df["time"], weather_df["temp_f"], color="#c2410c", label="Temp (F)")
    weather_ax.plot(weather_df["time"], weather_df["humidity"], color="#0369a1", label="Humidity (%)")
    weather_ax.plot(weather_df["time"], weather_df["wx_stream"], color="#15803d", linewidth=2, label="WX stream")
    weather_ax.plot(weather_df["time"], weather_df["wx_sum_running"], color="#7c3aed", linewidth=2, label="Running WX_SUM")
    weather_ax.axvspan(weather_metrics["window_start"], weather_metrics["settle"], color="#e5e7eb", alpha=0.35)
    weather_ax.axvline(weather_metrics["settle"], color="black", linestyle="--", linewidth=1)
    weather_ax.set_title(
        f"Weather | WX_SPOT now={weather_metrics['wx_spot_now']:.1f} | "
        f"settle proxy={weather_metrics['wx_spot_settle']:.1f} | WX_SUM={weather_metrics['wx_sum']:.1f}"
    )
    weather_ax.legend(loc="upper left", ncol=4)

    tide_ax = axes[1]
    tide_ax.plot(tide_df["time"], tide_df["level_m"], color="#0f172a", label="Observed tide level (m)")
    if projection_df is not None and not projection_df.empty:
        tide_ax.plot(projection_df["time"], projection_df["level_m"], color="#dc2626", linestyle="--", label="Projected level")
    tide_ax2 = tide_ax.twinx()
    tide_ax2.bar(tide_df["time"], tide_df["swing_payoff"], width=0.008, color="#16a34a", alpha=0.3, label="Swing payoff")
    tide_ax2.plot(tide_df["time"], tide_df["swing_running"], color="#7c3aed", linewidth=2, label="Running TIDE_SWING")
    tide_ax.axvspan(tide_metrics["window_start"], tide_metrics["settle"], color="#e5e7eb", alpha=0.35)
    tide_ax.axvline(tide_metrics["settle"], color="black", linestyle="--", linewidth=1)
    projected_text = "n/a" if tide_metrics["tide_spot_projected"] is None else f"{tide_metrics['tide_spot_projected']:.1f}"
    tide_ax.set_title(
        f"Tide | latest={tide_metrics['latest_level']:.3f}m | raw spot={tide_metrics['tide_spot_raw']:.1f} | "
        f"projected spot={projected_text} | observed swing={tide_metrics['observed_swing_sum']:.1f}"
    )
    tide_ax.set_ylabel("mAOD")
    tide_ax2.set_ylabel("Swing")
    tide_lines, tide_labels = tide_ax.get_legend_handles_labels()
    swing_lines, swing_labels = tide_ax2.get_legend_handles_labels()
    tide_ax.legend(tide_lines + swing_lines, tide_labels + swing_labels, loc="upper left", ncol=4)

    if has_flights:
        flight_ax = axes[2]
        if flights_df is not None and not flights_df.empty:
            hourly = flights_df.groupby(["hour", "side"]).size().unstack(fill_value=0)
            for column, color in (("arrival", "#2563eb"), ("departure", "#dc2626")):
                if column in hourly.columns:
                    flight_ax.plot(hourly.index, hourly[column], marker="o", linewidth=2, color=color, label=column.title())
        flight_ax.set_title(
            f"Heathrow | 12h snapshot count={flight_metrics['count']:.0f} | "
            f"window={flight_metrics['start']:%H:%M} to {flight_metrics['end']:%H:%M}"
        )
        flight_ax.legend(loc="upper left")

    now = datetime.now(LONDON_TZ)
    fig.suptitle(f"AlphaBot live data streams | refreshed {now:%Y-%m-%d %H:%M:%S %Z}", fontsize=16)
    display(fig)
    plt.close(fig)


## Live Dashboard

Run the next cell to start live updates. Change `REFRESH_SECS` if you want a slower or faster poll rate. Stop it with the notebook interrupt button.


In [ ]:
REFRESH_SECS = 60

while True:
    clear_output(wait=True)
    try:
        render_dashboard()
        print(f"Refreshing every {REFRESH_SECS} seconds. Interrupt the kernel to stop.")
    except Exception as exc:
        print(f"Refresh failed at {datetime.now(LONDON_TZ):%Y-%m-%d %H:%M:%S %Z}: {exc}")
    time.sleep(REFRESH_SECS)
